# 04 — E3: does style bias survive on newer judges? (W3)

**Default judges (2026-11-14): `gemini-3.8-flash` and `claude-sonnet-4-5`** -- see `config.yaml`. Master plan: rerun all 100 STYLE+STYLE_BA pairs, 3 repeats each, temperature 0.1, bias score with sign.

## Before you run this on Colab
1. Open: `https://colab.research.google.com/github/m00nLeo/judging-the-judges-study/blob/main/notebooks/04_E3_style_bias_new_judges.ipynb`
2. Run the setup cell (clone + deps + `fetch_upstream.sh`).
3. Add `GOOGLE_API_KEY` and `ANTHROPIC_API_KEY` to Colab Secrets (see notebook 03 for exact steps).
4. Dry-run -> pilot (5 pairs) -> full (100 pairs x 3 repeats), one judge at a time via `JUDGE_NAME`.
5. Send back `results/e3_style_bias/*.jsonl` when done.

In [ ]:
# On Colab, uncomment and run this cell BEFORE the cells below.
# !git clone https://github.com/m00nLeo/judging-the-judges-study.git
# %cd judging-the-judges-study

# !pip install -q anthropic openai google-genai pyyaml
# !bash scripts/fetch_upstream.sh

In [ ]:
import sys
from pathlib import Path

import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"
RESULTS_DIR = PROJECT_ROOT / "results" / "e3_style_bias"

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
MAX_WORKERS = config["concurrency"]["max_workers"]

from src.experiments.style_bias import load_style_pairs, run_style_bias_experiment, summarize_style_bias
from src.experiments.common import CostModel
from src.judges.clients import build_judge

style_pairs = load_style_pairs(UPSTREAM / "data" / "custom" / "controlled_pairs.jsonl")
n_repeats = config["sampling"]["e3_style_bias"]["n_repeats"]
print(f"{len(style_pairs)} STYLE+STYLE_BA pairs, {n_repeats} repeats each -> {len(style_pairs)*n_repeats} calls per judge")

## Step 1 -- dry run (zero cost)

In [ ]:
dry_cache = RESULTS_DIR / "_dry_run_cache.jsonl"
dry_cache.unlink(missing_ok=True)
dry_judge = build_judge("dry-run", dry_run=True)
cache = run_style_bias_experiment(style_pairs, dry_judge, judge_name="dry-run", cache_path=dry_cache, n_repeats=n_repeats, dry_run=True)
summary = summarize_style_bias(style_pairs, cache, n_repeats=n_repeats)
assert summary["n_calls_found"] == summary["n_calls_expected"]
print("PASS:", summary)

## Step 2 -- pilot (REAL calls, 5 pairs)

In [ ]:
JUDGE_NAME = config["judges"]["e3_style_bias"][0]  # change index to try the other judge
price = config["cost_per_1k_tokens"][JUDGE_NAME]
cost_model = CostModel(cost_per_1k_input=price["input"], cost_per_1k_output=price["output"])
budget = config["budget"]["e3_style_bias_usd"]

judge = build_judge(JUDGE_NAME, dry_run=False)
pilot_cache_path = RESULTS_DIR / f"{JUDGE_NAME}_pilot.jsonl"

pilot_cache = run_style_bias_experiment(
    style_pairs[:5], judge, judge_name=JUDGE_NAME, cache_path=pilot_cache_path,
    n_repeats=n_repeats, cost_model=cost_model, budget_usd=budget, dry_run=False, max_workers=MAX_WORKERS,
)
print(summarize_style_bias(style_pairs[:5], pilot_cache, n_repeats=n_repeats))
print("Review cost above before Step 3.")

## Step 3 -- full (all pairs, all repeats)

In [ ]:
full_cache_path = RESULTS_DIR / f"{JUDGE_NAME}_full.jsonl"
full_cache = run_style_bias_experiment(
    style_pairs, judge, judge_name=JUDGE_NAME, cache_path=full_cache_path,
    n_repeats=n_repeats, cost_model=cost_model, budget_usd=budget, dry_run=False, max_workers=MAX_WORKERS,
)
result = summarize_style_bias(style_pairs, full_cache, n_repeats=n_repeats)
print(result)
print("\nMaster plan's acceptance criteria (notebook 02's plan, mục 8/E3):")
score = result["style_bias_score"]
if score is not None:
    if abs(score) >= 0.3:
        print(f"|score|={abs(score):.2f} >= 0.3: style bias is still prominent on this judge.")
    elif abs(score) < 0.1:
        print(f"|score|={abs(score):.2f} < 0.1: style bias has dropped a lot on this judge.")
    else:
        print(f"|score|={abs(score):.2f}: in between -- not a clean call either way.")
print(f"\nRepeat Steps 2-3 with JUDGE_NAME = config['judges']['e3_style_bias'][1] for the second judge.")
print("When both judges are done, send back results/e3_style_bias/*_full.jsonl.")